In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import math
import torch.nn as nn
import torch.nn.functional as F
from timm.models.layers import to_2tuple, trunc_normal_
from collections import OrderedDict
from torch.utils.data import DataLoader
from torchvision import transforms
import shutil
from datasets import load_dataset
from PIL import Image

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [3]:
def quantize(tensor: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:

    Q_MIN = -127.0
    Q_MAX = 127.0
    scale_view = scale.squeeze()

    quantized_tensor = tensor / scale_view

    quantized_tensor = torch.round(quantized_tensor)

    quantized_tensor = torch.clamp(quantized_tensor, Q_MIN, Q_MAX)

    return quantized_tensor


def dequantize(quantized_tensor_float: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:

    scale_view = scale.squeeze()

    dequantized_tensor = quantized_tensor_float * scale_view

    return dequantized_tensor

In [4]:
def softmax(x, dim=-1):

    max_x = torch.max(x, dim=dim, keepdim=True)[0]

    two_x = torch.pow(2.0, x - max_x)

    sum_two_x = torch.sum(two_x, dim=dim, keepdim=True)

    return two_x / sum_two_x

In [5]:
class FloatAttnMaskSoftmax(nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        self.softmax = nn.Softmax(dim=-1)
        self.num_heads = num_heads

    def forward(self, attn_float, mask, B_, N):
        attn = torch.round(attn_float)
        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
        # base-2 softmax via ln(2)
        attn = softmax(attn)
        return attn

In [36]:
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None,
                 out_features=None, act_layer=nn.ReLU):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features

        self.fuse_linear_bn1 = nn.Linear(in_features, hidden_features)
        self.fuse_linear_bn2 = nn.Linear(hidden_features, out_features)

        self.register_buffer("fuse_linear_bn1_output_scale", None)
        self.register_buffer("fuse_linear_bn2_output_scale", None)

    def forward(self, x):

        x = self.fuse_linear_bn1(x)
        x = quantize(x, self.fuse_linear_bn1_output_scale)

        x = nn.ReLU()(x)

        x = self.fuse_linear_bn2(x)
        x = quantize(x, self.fuse_linear_bn2_output_scale)

        return x

def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    B = windows.shape[0] // (H * W // window_size // window_size)
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x


class WindowAttention(nn.Module):

    def __init__(self, dim, window_size, num_heads, qkv_bias=True):

        super().__init__()
        self.dim = dim
        self.window_size = window_size  # Wh, Ww
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = (0.5 if num_heads == 3 else
                     0.5 if num_heads == 6 else
                     0.25 if num_heads == 12 else
                     0.25 if num_heads == 24 else
                     head_dim ** -0.5)

        self.fuse_qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)

        self.float_mask_softmax = FloatAttnMaskSoftmax(num_heads)

        self.register_buffer("fuse_qkv_output_scale", None)
        self.register_buffer("qk_output_scale", None)
        self.register_buffer("relative_position_bias", None)
        # self.register_buffer("softmax_input_scale", None)
        # self.register_buffer("softmax_output_scale", None)
        self.register_buffer("attn_v_output_scale", None)
        self.register_buffer("proj_output_scale", None)

    def forward(self, x, mask=None):

        B_, N, C = x.shape

        qkv = self.fuse_qkv(x)
        qkv = quantize(qkv, self.fuse_qkv_output_scale)

        qkv = qkv.reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # make torchscript happy (cannot use tensor as tuple)

        q = q * self.scale
        attn = torch.matmul(q, k.transpose(-2, -1))
        attn = attn + self.relative_position_bias
        attn = quantize(attn, self.qk_output_scale)

        # attn = attn + self.relative_position_bias
        # attn = torch.clamp(attn, -127, 127)

        # attn = dequantize(attn, self.softmax_input_scale)

        attn = self.float_mask_softmax(attn, mask, B_, N)

        power = torch.log2(attn)
        power = torch.round(power)
        power = torch.clamp(power, -127, 127)
        power = power + 7
        attn = 2 ** power
        attn = torch.clamp(attn, -127, 127)

        # attn = quantize(attn, self.softmax_output_scale)

        x = torch.matmul(attn, v)
        x = quantize(x, self.attn_v_output_scale)

        x = x.transpose(1, 2).reshape(B_, N, C)

        x = self.proj(x)
        x = quantize(x, self.proj_output_scale)

        return x

class SwinTransformerBlock(nn.Module):

    def __init__(self, dim, input_resolution, num_heads, window_size=7, shift_size=0,
                 mlp_ratio=4., qkv_bias=True, act_layer=nn.ReLU):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        if min(self.input_resolution) <= self.window_size:
            # if window size is larger than input resolution, we don't partition windows
            self.shift_size = 0
            self.window_size = min(self.input_resolution)
        assert 0 <= self.shift_size < self.window_size, "shift_size must in 0-window_size"

        self.attn = WindowAttention(
            dim, window_size=to_2tuple(self.window_size), num_heads=num_heads,
            qkv_bias=qkv_bias)

        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer)

        self.register_buffer("attn_mask", None)

        # self.input_norm1 = nn.Linear(dim, dim)
        # self.input_norm2 = nn.Linear(dim, dim)

        # self.register_buffer("input_norm1_output_scale", None)
        # self.register_buffer("shortcut_add1_scale", None)
        # self.register_buffer("x_add1_scale", None)
        # self.register_buffer("input_norm2_output_scale", None)
        # self.register_buffer("x_add2_scale", None)
        # self.register_buffer("x_mlp_add2_scale", None)

    def forward(self, x):
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"

        shortcut = x

        # x = self.input_norm1(x)
        # x = quantize(x, self.input_norm1_output_scale)

        x = x.view(B, H, W, C)

        # cyclic shift
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
            # partition windows
            x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C
        else:
            shifted_x = x
            # partition windows
            x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C

        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)  # nW*B, window_size*window_size, C

        # W-MSA/SW-MSA
        attn_windows = self.attn(x_windows, mask=self.attn_mask)  # nW*B, window_size*window_size, C

        # merge windows
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)

        # reverse cyclic shift
        if self.shift_size > 0:
            shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
            x = shifted_x

        x = x.view(B, H * W, C)
        # shortcut = quantize(shortcut, self.shortcut_add1_scale)
        # x = quantize(x, self.x_add1_scale)
        x = shortcut + x
        x = torch.clamp(x, -127.0, 127.0)

        # FFN
        # x = self.input_norm2(x)
        # x = quantize(x, self.input_norm2_output_scale)

        x_mlp = self.mlp(x)

        # x = quantize(x, self.x_add2_scale)
        # x_mlp = quantize(x_mlp, self.x_mlp_add2_scale)
        x = x + x_mlp
        x = torch.clamp(x, -127.0, 127.0)

        return x


class PatchMerging(nn.Module):

    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim

        self.input_norm = nn.Linear(4 * dim, 2 * dim, bias=True)
        # self.input_norm = nn.Linear(4 * dim, 4 * dim)

        # self.register_buffer("reduction_output_scale", None)
        self.register_buffer("input_norm_output_scale", None)

    def forward(self, x):

        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"
        assert H % 2 == 0 and W % 2 == 0, f"x size ({H}*{W}) are not even."


        x = x.view(B, H, W, C)

        x0 = x[:, 0::2, 0::2, :]  # B H/2 W/2 C
        x1 = x[:, 1::2, 0::2, :]  # B H/2 W/2 C
        x2 = x[:, 0::2, 1::2, :]  # B H/2 W/2 C
        x3 = x[:, 1::2, 1::2, :]  # B H/2 W/2 C
        x = torch.cat([x0, x1, x2, x3], -1)  # B H/2 W/2 4*C
        x = x.view(B, -1, 4 * C)  # B H/2*W/2 4*C

        x = self.input_norm(x)
        x = quantize(x, self.input_norm_output_scale)

        # x = self.reduction(x)
        # x = quantize(x, self.reduction_output_scale)

        return x


class BasicLayer(nn.Module):

    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, downsample=None):

        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.depth = depth

        # build blocks
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(dim=dim, input_resolution=input_resolution,
                                 num_heads=num_heads, window_size=window_size,
                                 shift_size=0 if (i % 2 == 0) else window_size // 2,
                                 mlp_ratio=mlp_ratio,
                                 qkv_bias=qkv_bias)
            for i in range(depth)])

        # patch merging layer
        if downsample is not None:
            self.downsample = downsample(input_resolution, dim=dim)
        else:
            self.downsample = None

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        if self.downsample is not None:
            x = self.downsample(x)
        return x


class PatchEmbed(nn.Module):

    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        patches_resolution = [img_size[0] // patch_size[0], img_size[1] // patch_size[1]]
        self.img_size = img_size
        self.patch_size = patch_size
        self.patches_resolution = patches_resolution

        self.in_chans = in_chans
        self.embed_dim = embed_dim

        self.fused_proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

        self.register_buffer("fused_proj_output_scale", None)

    def forward(self, x):
        B, C, H, W = x.shape
        # FIXME look at relaxing size constraints
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."

        x = self.fused_proj(x)

        x = x.flatten(2).transpose(1, 2)
        x = quantize(x, self.fused_proj_output_scale)

        return x


class SwinTransformer(nn.Module):

    def __init__(self, img_size=224, patch_size=4, in_chans=3, num_classes=1000,
                 embed_dim=96, depths=[2, 2, 6, 2], num_heads=[3, 6, 12, 24],
                 window_size=7, mlp_ratio=4., qkv_bias=True, **kwargs):
        super().__init__()

        self.num_classes = num_classes
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.num_features = int(embed_dim * 2 ** (self.num_layers - 1))
        self.mlp_ratio = mlp_ratio

        # split image into non-overlapping patches
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim)
        patches_resolution = self.patch_embed.patches_resolution
        self.patches_resolution = patches_resolution

        # build layers
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            layer = BasicLayer(dim=int(embed_dim * 2 ** i_layer),
                               input_resolution=(patches_resolution[0] // (2 ** i_layer),
                                                 patches_resolution[1] // (2 ** i_layer)),
                               depth=depths[i_layer],
                               num_heads=num_heads[i_layer],
                               window_size=window_size,
                               mlp_ratio=self.mlp_ratio,
                               qkv_bias=qkv_bias,
                               downsample=PatchMerging if (i_layer < self.num_layers - 1) else None)
            self.layers.append(layer)

        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(OrderedDict([("fc", nn.Linear(self.num_features, num_classes))])) if num_classes > 0 else nn.Identity()

        self.input_norm = nn.Linear(self.num_features, self.num_features)

        self.register_buffer("input_scale", None)
        self.register_buffer("input_norm_output_scale", None)
        self.register_buffer("output_scale", None)

    def forward_features(self, x):
        x = quantize(x, self.input_scale)
        x = self.patch_embed(x)

        for layer in self.layers:
            x = layer(x)

        x = self.input_norm(x)
        x = quantize(x, self.input_norm_output_scale)

        x = self.avgpool(x.transpose(1, 2))  # B C 1
        x = torch.flatten(x, 1)
        return x

    def forward(self, x):
        x = self.forward_features(x)

        x = self.head(x)
        x = dequantize(x, self.output_scale)

        return x

In [37]:
# 2. Create your custom implementation
custom_model = SwinTransformer(
    img_size=224,
    patch_size=4,
    in_chans=3,
    num_classes=1000,
    embed_dim=96,
    depths=[2, 2, 6, 2],
    num_heads=[3, 6, 12, 24],
)

In [8]:
orig_state = torch.load("/content/drive/MyDrive/BN+softmax+ReLU+fusedBN+foldedBN+fusedReLU+PTQInference7.pth")

In [38]:
for name, param in orig_state.items():
    # Check if the name corresponds to an observer's scale
    if "scale" in name or "relative_position_bias" in name or "attn_mask" in name and "table" not in name:
        # Construct the path to the observer module
        parts = name.split('.')
        current_module = custom_model
        module_found = True
        for i, part in enumerate(parts[:-1]):
            if hasattr(current_module, part):
                current_module = getattr(current_module, part)
            else:
                # If a part of the path doesn't exist, this key is not applicable
                module_found = False
                break

        if module_found:
            attr_name = parts[-1] # This should be 'scale'
            if hasattr(current_module, attr_name):
                # Directly assign the parameter from the loaded state to the model's buffer
                # This will resize the buffer if necessary, fixing the shape mismatch.
                with torch.no_grad(): # Ensure this operation doesn't track gradients
                    # Ensure device consistency by moving param to buffer's device if different
                    setattr(current_module, attr_name, param)
            else:
                print(f"Warning: Attribute '{attr_name}' not found in module '{'.'.join(parts[:-1])}' for key '{name}'. Skipping.")
        else:
            print(f"Warning: Module path for key '{name}' not fully resolvable. Skipping.")


# Load into your model
missing, unexpected = custom_model.load_state_dict(orig_state, strict=False)


# Optionally inspect for any mismatches
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

Missing keys: ['layers.0.blocks.0.input_norm1.weight', 'layers.0.blocks.0.input_norm1.bias', 'layers.0.blocks.0.input_norm2.weight', 'layers.0.blocks.0.input_norm2.bias', 'layers.0.blocks.1.input_norm1.weight', 'layers.0.blocks.1.input_norm1.bias', 'layers.0.blocks.1.input_norm2.weight', 'layers.0.blocks.1.input_norm2.bias', 'layers.1.blocks.0.input_norm1.weight', 'layers.1.blocks.0.input_norm1.bias', 'layers.1.blocks.0.input_norm2.weight', 'layers.1.blocks.0.input_norm2.bias', 'layers.1.blocks.1.input_norm1.weight', 'layers.1.blocks.1.input_norm1.bias', 'layers.1.blocks.1.input_norm2.weight', 'layers.1.blocks.1.input_norm2.bias', 'layers.2.blocks.0.input_norm1.weight', 'layers.2.blocks.0.input_norm1.bias', 'layers.2.blocks.0.input_norm2.weight', 'layers.2.blocks.0.input_norm2.bias', 'layers.2.blocks.1.input_norm1.weight', 'layers.2.blocks.1.input_norm1.bias', 'layers.2.blocks.1.input_norm2.weight', 'layers.2.blocks.1.input_norm2.bias', 'layers.2.blocks.2.input_norm1.weight', 'layers.2

In [10]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: 
Token is valid (permission: fineGrained).
The token `swin` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `swin`


In [39]:
# ----------------------------------------------------------------------------
# 1) Load ImageNet-1k training set (streaming mode)
# ----------------------------------------------------------------------------
hf_dataset_train = load_dataset("ILSVRC/imagenet-1k", split="train", streaming=True)
hf_dataset_val = load_dataset("ILSVRC/imagenet-1k", split="validation", streaming=True)


train_stream = hf_dataset_train.take(100_000)
val_stream   = hf_dataset_train.skip(100_000).take(15_000)
test_stream = hf_dataset_val.take(10_000)


# ----------------------------------------------------------------------------
# 2) Preprocessing pipeline
# ----------------------------------------------------------------------------
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def collate_fn(batch):
    """Convert HF samples into tensors"""
    images, labels = [], []
    for sample in batch:
        img = sample["image"]
        if not isinstance(img, Image.Image):  # if numpy array
            img = Image.fromarray(img)
        img = img.convert("RGB")  # <-- ensure 3 channels
        images.append(transform(img))
        labels.append(sample["label"])
    return torch.stack(images), torch.tensor(labels)


# ----------------------------------------------------------------------------
# 3) Wrap datasets with DataLoader
# ----------------------------------------------------------------------------
train_loader = DataLoader(train_stream, batch_size=64, collate_fn=collate_fn)
val_loader   = DataLoader(val_stream,   batch_size=64, collate_fn=collate_fn)
test_loader  = DataLoader(test_stream, batch_size=64, collate_fn=collate_fn)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    top1, top5, total, total_loss = 0, 0, 0, 0.0

    with torch.no_grad():
        for i, (images, labels) in enumerate(dataloader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Top-1
            _, pred1 = outputs.topk(1, dim=1)
            top1 += (pred1.squeeze() == labels).sum().item()

            # Top-5
            _, pred5 = outputs.topk(5, dim=1)
            top5 += sum([labels[j].item() in pred5[j] for j in range(labels.size(0))])

            total += labels.size(0)

            if (i+1) % 1 == 0:
                print(f"[Val Step {i+1}] "
                      f"Loss: {total_loss/(i+1):.4f} | "
                      f"Top-1: {100.*top1/total:.2f}% | "
                      f"Top-5: {100.*top5/total:.2f}%")

    return 100.*top1/total, 100.*top5/total, total_loss/(i+1)

# ----------------------------------------------------------------------------
# 5) Train the model
# ----------------------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
custom_model.to(device)

criterion = nn.CrossEntropyLoss()

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

In [40]:
test_top1, test_top5, test_loss = evaluate(
    custom_model, train_loader, criterion, device
)
print(f"\nPTQ Model Results:")
print(f"Top-1: {test_top1:.2f}% | Top-5: {test_top5:.2f}%")

[Val Step 1] Loss: 0.3667 | Top-1: 90.62% | Top-5: 96.88%
[Val Step 2] Loss: 0.5085 | Top-1: 87.50% | Top-5: 96.09%
[Val Step 3] Loss: 0.5600 | Top-1: 86.98% | Top-5: 96.35%
[Val Step 4] Loss: 0.5177 | Top-1: 87.89% | Top-5: 97.27%
[Val Step 5] Loss: 0.5436 | Top-1: 86.56% | Top-5: 97.19%
[Val Step 6] Loss: 0.5422 | Top-1: 86.72% | Top-5: 97.14%
[Val Step 7] Loss: 0.5211 | Top-1: 87.05% | Top-5: 97.54%
[Val Step 8] Loss: 0.5278 | Top-1: 87.11% | Top-5: 97.27%
[Val Step 9] Loss: 0.5350 | Top-1: 86.63% | Top-5: 97.22%
[Val Step 10] Loss: 0.5675 | Top-1: 85.94% | Top-5: 97.03%
[Val Step 11] Loss: 0.5742 | Top-1: 85.80% | Top-5: 97.16%
[Val Step 12] Loss: 0.5855 | Top-1: 85.29% | Top-5: 97.14%
[Val Step 13] Loss: 0.5816 | Top-1: 85.34% | Top-5: 97.00%
[Val Step 14] Loss: 0.5714 | Top-1: 85.60% | Top-5: 97.10%
[Val Step 15] Loss: 0.5901 | Top-1: 85.00% | Top-5: 96.88%
[Val Step 16] Loss: 0.5751 | Top-1: 85.25% | Top-5: 96.88%
[Val Step 17] Loss: 0.6049 | Top-1: 84.83% | Top-5: 96.42%
[Val S

KeyboardInterrupt: 

In [41]:
test_top1, test_top5, test_loss = evaluate(
    custom_model, test_loader, criterion, device
)
print(f"\nPTQ Model Results:")
print(f"Top-1: {test_top1:.2f}% | Top-5: {test_top5:.2f}%")

[Val Step 1] Loss: 1.1033 | Top-1: 73.44% | Top-5: 92.19%
[Val Step 2] Loss: 1.1998 | Top-1: 71.88% | Top-5: 89.84%
[Val Step 3] Loss: 1.1532 | Top-1: 72.92% | Top-5: 90.62%
[Val Step 4] Loss: 1.1714 | Top-1: 73.05% | Top-5: 89.84%
[Val Step 5] Loss: 1.1650 | Top-1: 73.12% | Top-5: 90.00%
[Val Step 6] Loss: 1.2146 | Top-1: 72.14% | Top-5: 89.58%
[Val Step 7] Loss: 1.2613 | Top-1: 71.88% | Top-5: 89.73%
[Val Step 8] Loss: 1.2228 | Top-1: 72.46% | Top-5: 90.23%
[Val Step 9] Loss: 1.2722 | Top-1: 71.35% | Top-5: 89.24%
[Val Step 10] Loss: 1.2937 | Top-1: 70.78% | Top-5: 88.59%
[Val Step 11] Loss: 1.3036 | Top-1: 70.31% | Top-5: 88.35%
[Val Step 12] Loss: 1.2805 | Top-1: 70.44% | Top-5: 88.41%
[Val Step 13] Loss: 1.2677 | Top-1: 70.67% | Top-5: 88.58%
[Val Step 14] Loss: 1.2902 | Top-1: 70.54% | Top-5: 88.62%
[Val Step 15] Loss: 1.2923 | Top-1: 70.52% | Top-5: 88.44%
[Val Step 16] Loss: 1.2991 | Top-1: 70.31% | Top-5: 88.18%
[Val Step 17] Loss: 1.3356 | Top-1: 69.67% | Top-5: 87.59%
[Val S

KeyboardInterrupt: 

In [42]:
torch.save(custom_model.state_dict(), "final_weights10.pth")

drive_path = "/content/drive/MyDrive/final_weights10.pth"
shutil.copy("final_weights10.pth", drive_path)

'/content/drive/MyDrive/final_weights10.pth'

In [43]:
orig_state = torch.load("/content/drive/MyDrive/final_weights10.pth")
with open("final_weights10.txt", "w") as f:
    for name, param in orig_state.items():
        f.write(f"{name}: {param}\n\n")

drive_path = "/content/drive/MyDrive/final_weights10.txt"
shutil.copy("final_weights10.txt", drive_path)

'/content/drive/MyDrive/final_weights10.txt'

In [ ]:
import torch

def convert_swin_weights_types(input_model_path, output_model_path):

    print(f"Loading model from {input_model_path}...")
    state_dict = torch.load(input_model_path, map_location='cpu')

    new_state_dict = {}

    for key, value in state_dict.items():
        if not torch.is_tensor(value):
            new_state_dict[key] = value
            continue

        if key.endswith(".bias"):
            new_state_dict[key] = value.round().to(torch.int32)
            print(f"Converted to INT32: {key}")

        elif key.endswith(".weight") or "attn_mask" in key or "relative_position_bias" in key:
            new_state_dict[key] = value.round().to(torch.int8)
            print(f"Converted to INT8: {key}")

        else:
            new_state_dict[key] = value
            print(f"Kept AS IS: {key}")

    print(f"Saving converted model to {output_model_path}...")
    torch.save(new_state_dict, output_model_path)
    print("Done!")

input_path = "/content/drive/MyDrive/final_weights3.pth"

output_path = "/content/drive/MyDrive/final_weights3_int.pth"

convert_swin_weights_types(input_path, output_path)